# Анализ продаж интернет-магазина за 2025 год

**Тип проекта:** учебный кейс на синтетических данных. Все товары, покупатели, заказы и финансовые показатели сгенерированы.

## Бизнес-задача

Определить, какие категории и каналы формируют выручку и валовую прибыль, как показатели меняются в течение года, где отличаются отмены и возвраты, а также проверить качество данных.

## Вопросы
1. Как менялись выручка, валовая прибыль и маржа по месяцам?
2. Какие категории и товары лидируют по выручке и прибыли?
3. Как отличаются каналы продаж по выручке, среднему чеку и марже?
4. Каковы доли отмен и возвратов?
5. Можно ли доверять данным после базовых проверок качества?

**Правило:** выручка считается только по заказам со статусом `Доставлен`. Валовая прибыль = выручка после скидки − себестоимость товара. Прочие расходы не включены.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Поддерживает запуск из корня проекта и из каталога notebooks.
ROOT = Path.cwd()
if not (ROOT / "data" / "raw").exists() and (ROOT.parent / "data" / "raw").exists():
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw"

customers = pd.read_csv(RAW / "customers.csv", parse_dates=["signup_date"])
products = pd.read_csv(RAW / "products.csv")
orders = pd.read_csv(RAW / "orders.csv", parse_dates=["order_date"])
items = pd.read_csv(RAW / "order_items.csv")

print(f"Покупатели: {len(customers):,}")
print(f"Товары: {len(products):,}")
print(f"Заказы: {len(orders):,}")
print(f"Строки заказов: {len(items):,}")

## 1. Проверка качества данных
Проверяем уникальность ключей, ссылочную целостность и допустимые диапазоны количеств и скидок.

In [ ]:
quality_checks = pd.DataFrame([
    {"check": "duplicate customer_id", "issues": int(customers.customer_id.duplicated().sum())},
    {"check": "duplicate product_id", "issues": int(products.product_id.duplicated().sum())},
    {"check": "duplicate order_id", "issues": int(orders.order_id.duplicated().sum())},
    {"check": "duplicate order_item_id", "issues": int(items.order_item_id.duplicated().sum())},
    {"check": "orders without customer", "issues": int((~orders.customer_id.isin(customers.customer_id)).sum())},
    {"check": "items without order", "issues": int((~items.order_id.isin(orders.order_id)).sum())},
    {"check": "items without product", "issues": int((~items.product_id.isin(products.product_id)).sum())},
    {"check": "non-positive quantity", "issues": int((items.quantity <= 0).sum())},
    {"check": "invalid discount", "issues": int(((items.discount_pct < 0) | (items.discount_pct > 1)).sum())},
])
display(quality_checks)
assert quality_checks.issues.sum() == 0, "Есть ошибки качества: проверьте проверки выше."

## 2. Подготовка аналитической таблицы
Соединяем строки заказа с заказами, товарами и покупателями. Рассчитываем выручку после скидки, себестоимость и валовую прибыль.

In [ ]:
facts = (items
    .merge(orders, on="order_id", how="left", validate="many_to_one")
    .merge(products[["product_id", "product_name", "category", "unit_cost"]], on="product_id", how="left", validate="many_to_one")
    .merge(customers[["customer_id", "region", "city"]], on="customer_id", how="left", validate="many_to_one"))
facts["month"] = facts.order_date.dt.to_period("M").astype(str)
facts["gross_sales"] = facts.quantity * facts.unit_price
facts["discount_amount"] = facts.gross_sales * facts.discount_pct
facts["net_revenue"] = facts.gross_sales - facts.discount_amount
facts["cogs"] = facts.quantity * facts.unit_cost
facts["gross_profit"] = facts.net_revenue - facts.cogs

delivered = facts[facts.order_status.eq("Доставлен")].copy()
display(delivered[["order_id", "order_date", "product_name", "quantity", "net_revenue", "gross_profit"]].head())

## 3. Основные KPI
Финансовые показатели считаем только по доставленным заказам. Отмены и возвраты изучаем отдельными операционными метриками.

In [ ]:
order_kpis = (delivered.groupby(["order_id", "customer_id", "order_date", "sales_channel"], as_index=False)
    .agg(net_revenue=("net_revenue", "sum"), gross_profit=("gross_profit", "sum"), units=("quantity", "sum")))
order_kpis["month"] = order_kpis.order_date.dt.to_period("M").astype(str)
status_counts = orders.order_status.value_counts()
revenue = delivered.net_revenue.sum()
gross_profit = delivered.gross_profit.sum()
unique_buyers = order_kpis.customer_id.nunique()
customer_orders = order_kpis.groupby("customer_id").order_id.nunique()
repeat_buyers = int((customer_orders >= 2).sum())

kpis = pd.DataFrame([
    ["Всего заказов", len(orders), "шт."],
    ["Доставленные заказы", int(status_counts.get("Доставлен", 0)), "шт."],
    ["Выручка после скидок", round(revenue, 2), "₽"],
    ["Валовая прибыль", round(gross_profit, 2), "₽"],
    ["Валовая маржа", round(gross_profit / revenue, 4), "доля"],
    ["Средний чек", round(order_kpis.net_revenue.mean(), 2), "₽"],
    ["Уникальные покупатели", unique_buyers, "чел."],
    ["Доля повторных покупателей", round(repeat_buyers / unique_buyers, 4), "доля"],
    ["Доля отмен", round(status_counts.get("Отменён", 0) / len(orders), 4), "доля"],
], columns=["metric", "value", "unit"])
display(kpis)

## 4. Динамика по месяцам
Сравниваем выручку и валовую маржу. Важно не путать рост продаж с ростом прибыльности.

In [ ]:
monthly = (order_kpis.groupby("month", as_index=False)
    .agg(delivered_orders=("order_id", "nunique"), net_revenue=("net_revenue", "sum"), gross_profit=("gross_profit", "sum")))
monthly["gross_margin_pct"] = monthly.gross_profit / monthly.net_revenue
monthly["aov"] = monthly.net_revenue / monthly.delivered_orders
display(monthly.round(3))

fig, ax1 = plt.subplots(figsize=(11, 5))
ax1.plot(monthly.month, monthly.net_revenue, marker="o", linewidth=2, label="Выручка")
ax1.set_ylabel("Выручка, ₽")
ax1.tick_params(axis="x", rotation=45)
ax2 = ax1.twinx()
ax2.plot(monthly.month, monthly.gross_margin_pct * 100, marker="s", linestyle="--", label="Валовая маржа")
ax2.set_ylabel("Валовая маржа, %")
ax1.set_title("Выручка и валовая маржа по месяцам")
ax1.grid(axis="y", alpha=.25)
fig.tight_layout()
plt.show()

## 5. Категории и товары
Смотрим на выручку и валовую прибыль вместе. Категория с максимальной выручкой может уступать по марже.

In [ ]:
category = (delivered.groupby("category", as_index=False)
    .agg(units_sold=("quantity", "sum"), net_revenue=("net_revenue", "sum"), gross_profit=("gross_profit", "sum"), orders=("order_id", "nunique")))
category["gross_margin_pct"] = category.gross_profit / category.net_revenue
category["revenue_share_pct"] = category.net_revenue / category.net_revenue.sum()
category = category.sort_values("net_revenue", ascending=False)
display(category.round(3))

cat_plot = category.sort_values("net_revenue")
y = np.arange(len(cat_plot))
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(y - .18, cat_plot.net_revenue / 1000, height=.35, label="Выручка, тыс. ₽")
ax.barh(y + .18, cat_plot.gross_profit / 1000, height=.35, label="Валовая прибыль, тыс. ₽")
ax.set_yticks(y, cat_plot.category)
ax.set_xlabel("тыс. ₽")
ax.set_title("Выручка и валовая прибыль по категориям")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

top_products = (delivered.groupby(["product_id", "product_name", "category"], as_index=False)
    .agg(units_sold=("quantity", "sum"), net_revenue=("net_revenue", "sum"), gross_profit=("gross_profit", "sum"))
    .sort_values("net_revenue", ascending=False))
display(top_products.head(10).round(2))

## 6. Каналы и регионы
Для каждого канала считаем выручку, валовую прибыль, маржу, средний чек, а статусы заказов рассматриваем отдельно. Без трафика и расходов на рекламу нельзя посчитать конверсию или ROAS.

In [ ]:
channels = (order_kpis.groupby("sales_channel", as_index=False)
    .agg(delivered_orders=("order_id", "nunique"), net_revenue=("net_revenue", "sum"), gross_profit=("gross_profit", "sum")))
channels["aov"] = channels.net_revenue / channels.delivered_orders
channels["gross_margin_pct"] = channels.gross_profit / channels.net_revenue
all_orders_by_channel = orders.groupby("sales_channel").size().rename("all_orders")
status_pivot = pd.crosstab(orders.sales_channel, orders.order_status)
channels = channels.merge(all_orders_by_channel, left_on="sales_channel", right_index=True)
channels = channels.merge(status_pivot, left_on="sales_channel", right_index=True, how="left")
channels["cancel_rate_pct"] = channels.get("Отменён", 0) / channels.all_orders
display(channels.sort_values("net_revenue", ascending=False).round(3))

regions = (delivered.groupby("region", as_index=False)
    .agg(delivered_orders=("order_id", "nunique"), buyers=("customer_id", "nunique"), net_revenue=("net_revenue", "sum"), gross_profit=("gross_profit", "sum")))
regions["gross_margin_pct"] = regions.gross_profit / regions.net_revenue
regions["revenue_per_buyer"] = regions.net_revenue / regions.buyers
display(regions.sort_values("net_revenue", ascending=False).round(2))

## 7. Выводы и ограничения

При запуске на текущей сгенерированной выборке анализ показывает:

- выручка заметно увеличивается к концу года;
- категория «Электроника» формирует наибольшую долю выручки, но имеет более низкую валовую маржу, чем «Аксессуары»;
- в ноябре и декабре рост выручки сопровождается снижением валовой маржи и более высокой средней скидкой;
- каналы различаются и по размеру выручки, и по отменам/возвратам — их не стоит ранжировать по одной метрике.

**Что нельзя утверждать:** данные синтетические, поэтому выводы не описывают реальный магазин. Совпадение роста скидки со снижением маржи не доказывает, что скидка — единственная причина. Для расчёта чистой прибыли нужны расходы на маркетинг, комиссии и логистику; для конверсии — данные о посещениях/показах.

Следующий бизнес-шаг: проверить экономику скидок в сезон, выяснить причины возвратов и добавить переменные расходы по каналам.